In [ ]:
include_dir = "/vol/aibn133/data1/aercolino/SOFTWARE/SN_from_grids/JuliaScripts/"

include(include_dir*"defs.jl")
include(include_dir*"Plot_defs.jl")

moving_average(vs,n) = [sum(@view vs[i:(i+n-1)])/n for i in 1:(length(vs)-(n-1))]


Plots the models from hjin/drad on the Xc/Mco plane


In [ ]:

CRITERION = "M16"
CALIBRATOR = "N20_old" #"Z9.6_S19.8" # "Z9.6_W20"
ix_single = get_ix_imploders(f_singleCC, fexp_single, "logM", CRITERION, CALIBRATOR)
ix_drad = get_ix_imploders(f_HeS_DRAD_z0p015, fexp_DRAD, "M", CRITERION, CALIBRATOR)
ix_HeS = get_ix_imploders(f_HeSCC, fexp_HeS, "logM", CRITERION, CALIBRATOR)


FIGURE = Figure(size=(500,500))
ax = plot_settings(FIGURE[1, 1]; 
    xlabel = L"M_\mathrm{CO} \ [M_\odot]", 
    ylabel=L"X_\mathrm{C}", 
    title ="",
    xticks=(0:2:22.5, [L"%$(i)" for i in 0:2:22]), yticks=(0.:0.05:0.5, [L"%$(@sprintf(\"%.2f\",i))" for i in 0.00:0.05:0.50]),
    ymticks=0:0.01:0.5)


ylims!(0.18,0.41)
xlims!(2,12.5)
vars = Dict("BG_1"=>Dict(),  
            "BG_2" => Dict(),
            "s_hjin_0"=> Dict(),        
            "s_hjin_150"=> Dict(),        
            "s_hjin_300"=> Dict(),
            "s_CC"=>Dict(),
            "HeS_CC"=>Dict()
            )
for Z in keys(Zs)
    drad_z = @sprintf("drad_%.3f", Z)
    vars[drad_z] = Dict()
end
            
when = "ini_BURN_C"
#when = "end_BURN_He"
MASS_LIMIT_hjin = 60.0
for key in ["Cc", "M_co_core", "M_he_core"]
    filterout = 10. .^ master_f[!, "logM"].< (MASS_LIMIT_hjin)
    
    
    vars["BG_1"][key]   = (master_f[!, key*"_1_"*when*"_1"])[filterout]
    @printf("length : %d -> %d ", length(filterout), length(vars["BG_1"][key]))
    
    filterout = 10. .^ master_f[!, "logM"] .*master_f[!, "q"] .< (MASS_LIMIT_hjin)    
    vars["BG_2"][key]   = (master_f[!, key*"_2_"*when*"_2"])[filterout]

    # filterout = 10. .^ f_single_hjin_0[!, "logM"].< (MASS_LIMIT_hjin)    
    # vars["s_hjin_0"][key] = (f_single_hjin_0[!, key*"_"*when])[filterout]  

    filterout = 10. .^ f_single_hjin_150[!, "logM"].< (MASS_LIMIT_hjin)    
    vars["s_hjin_150"][key] = (f_single_hjin_150[!, key*"_"*when])[filterout]  

    vars["s_CC"][key] = f_singleCC[!, key*"_"*when] 

    vars["HeS_CC"][key] =  f_HeSCC[!, key*"_"*when]  

    for Z in keys(Zs)
        drad_z=@sprintf("drad_%.3f", Z)
        vars[drad_z][key] = f_HeS_DRAD[Z][!, key*"_"*when]
    end
end
for Z in keys(Zs)
    drad_z=@sprintf("drad_%.3f", Z)
    vars[drad_z]["xi2p5e_end"] = f_HeS_DRAD[Z][!, "xi2p5e_end"]
    vars[drad_z]["Mi"] = f_HeS_DRAD[Z][!, "M"]
end

cticks = 0.1:0.05:2
cticks_str = [@sprintf("%.1f", 10. ^ ctick) for ctick in cticks]
cticks_labels = [L"%$(i)" for i in cticks_str]

scatter!(ax, PS.M, PS.X, color = "lightgray", marker = :rect,  markersize=4, alpha = 1)

myplot2=scatter!(ax, vars["BG_2"]["M_co_core"],   vars["BG_2"]["Cc"],   color=:orange, marker = :circle, markersize = 4, alpha = 1, label = "hjin (Binary - Secondary)")
myplot1=scatter!(ax, vars["BG_1"]["M_co_core"],   vars["BG_1"]["Cc"],   color=:green, marker = :circle, markersize = 4, alpha = 0.7, label = "hjin (Binary - Primariy)")
#lines!(ax, vars["s_hjin_0"]["M_co_core"], vars["s_hjin_0"]["Cc"], color=:gray, linewidth = 5, alpha = 1, label = "hjin (Single - non-rot.)")
#lines!(ax, vars["s_hjin_150"]["M_co_core"], vars["s_hjin_150"]["Cc"], color=:gray, linewidth = 5, alpha = 0.7, label = "hjin (Single - 20% crit)")
lines!(ax, vars["s_CC"]["M_co_core"], vars["s_CC"]["Cc"], color=:red, linewidth = 5, alpha = 0.7, label = "Single to CC - 20% crit")
lines!(ax, vars["HeS_CC"]["M_co_core"], vars["HeS_CC"]["Cc"], color=:lightblue, linewidth = 5, alpha = .7, label = "Single HeS to CC")
# lines!(ax, vars["s_hjin_300"]["M_co_core"], vars["s_hjin_300"]["Cc"], color=:lightgray, linewidth = 5, alpha = 1, label = "hjin (single - 40% crit)")
for Z in keys(Zs)
    drad_z=@sprintf("drad_%.3f", Z)
    lines!(ax, vars[drad_z]["M_co_core"], vars[drad_z]["Cc"], color=:black,   linewidth = Z == 0.015 ? 1.5 : 0.5 , alpha = 0.5, label = "Single HeS to CC (AD+23)")
    # scatter!(ax, vars[drad_z]["M_co_core"], vars[drad_z]["Cc"]*1., 
    #         color=:red,   marker = 'o', alpha = 0.2)

    scatter!(ax, vars[drad_z]["M_co_core"][ix_drad], vars[drad_z]["Cc"][ix_drad], color = :black, marker = '+', markersize = 29)
    scatter!(ax, vars[drad_z]["M_co_core"], vars[drad_z]["Cc"], color = :black, marker = :circle, markersize = 5)
    #scatter!(ax, vars[drad_z]["M_co_core"], vars[drad_z]["Cc"], color = :brown, marker = :circle, markersize = 5)

end

scatter!(ax, vars["HeS_CC"]["M_co_core"][ix_HeS], vars["HeS_CC"]["Cc"][ix_HeS], color = :cyan, marker = '+',   markersize = 29)
scatter!(ax, vars["HeS_CC"]["M_co_core"], vars["HeS_CC"]["Cc"], color = :cyan, marker = :circle,   markersize = 5)
#scatter!(ax, vars["HeS_CC"]["M_co_core"], vars["HeS_CC"]["Cc"], color = :cyan, marker = :circle,   markersize = 5)
scatter!(ax, vars["s_CC"]["M_co_core"][ix_single], vars["s_CC"]["Cc"][ix_single], color = :brown, marker = '+',  markersize = 29)
scatter!(ax, vars["s_CC"]["M_co_core"], vars["s_CC"]["Cc"], color = :brown, marker = :circle,  markersize = 5)
# Colorbar(FIGURE[1, 1], myplot, size =25, label = L"M_\mathrm{1,i} \ [M_\odot]", 
#             ticks=(cticks, cticks_labels), vertical=false, ticklabelrotation=pi/2
# )

axislegend(ax,  framevisible=true)

save("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/Xc_Mco.pdf", FIGURE)
FIGURE








In [ ]:
f_singleCC = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/aerk/singleCC/CORR_GRID_single_0p2_CC.dat",  DataFrame, delim=' ',ignorerepeated=true)
f_HeSCC = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/aerk/singleCC/HeS/CORR_GRID_single_0p2_CC.dat",  DataFrame, delim=' ',ignorerepeated=true)
f_HeS_DRAD_z0p015 = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/drad/single_he_stars/CORR_DRAD_0.015.dat",  DataFrame, delim=' ',ignorerepeated=true)
fexp_single = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/aerk/singleCC/EXP_PROP_AD23.data",  DataFrame, delim=' ',ignorerepeated=true)
fexp_HeS    = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/aerk/singleCC/HeS/EXP_PROP_AD23.data",  DataFrame, delim=' ',ignorerepeated=true)
fexp_DRAD   = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/drad/single_he_stars/Z_0.015/EXP_PROP_AD23.data",  DataFrame, delim=' ',ignorerepeated=true)

function corr_file_reducer_CC(corr_file, exp_file; mass_column = "logM")
    corr_file_exp = copy(corr_file)

    for ix in range(1,length(corr_file[!, mass_column]))
        logM = corr_file[!, mass_column][ix] 
        if ! (logM in exp_file.logM)
            #@printf("logM = %.3f -> is in exp? %s", logM, logM in exp_file.logM)
            #@printf(" -> removing from CORR")
            (mass_column == "logM") && (corr_file_exp = filter(row -> (row.logM in exp_file.logM), corr_file ))
            (mass_column == "M")    && (corr_file_exp = filter(row -> (row.M in exp_file.logM), corr_file ))

            #@printf("\n")
        end
    end
    return corr_file_exp
end 
exp_corr_S = corr_file_reducer_CC(f_singleCC,        fexp_single; mass_column = "logM")
exp_corr_AD = corr_file_reducer_CC(f_HeS_DRAD_z0p015, fexp_DRAD;   mass_column = "M"   )
exp_corr_He = corr_file_reducer_CC(f_HeSCC,           fexp_HeS;    mass_column = "logM")
print("s: ", exp_corr_S.logM == fexp_single.logM, "\n")
print("d: ", exp_corr_AD.M == fexp_DRAD.logM, "\n")
print("h: ", exp_corr_He.logM == fexp_HeS.logM, "\n")
exp_corr_S[!, "E_exp"] = fexp_single.E_exp  
exp_corr_AD[!, "E_exp"] = fexp_DRAD.E_exp  
exp_corr_He[!, "E_exp"] = fexp_HeS.E_exp  
exp_corr_S[!, "M_ej"] = fexp_single.M_ej  
exp_corr_AD[!, "M_ej"] = fexp_DRAD.M_ej  
exp_corr_He[!, "M_ej"] = fexp_HeS.M_ej  
exp_corr_S[!, "xi25"] = fexp_single.xi25  
exp_corr_AD[!, "xi25"] = fexp_DRAD.xi25  
exp_corr_He[!, "xi25"] = fexp_HeS.xi25  
exp_corr_S[!, "mu4"] = fexp_single.mu4  
exp_corr_AD[!, "mu4"] = fexp_DRAD.mu4  
exp_corr_He[!, "mu4"] = fexp_HeS.mu4  
exp_corr_S[!, "M4"] = fexp_single.M4  
exp_corr_AD[!, "M4"] = fexp_DRAD.M4  
exp_corr_He[!, "M4"] = fexp_HeS.M4  
exp_corr_S[!, "Mni"] = fexp_single[!, "M(Ni56)"]  
exp_corr_AD[!, "Mni"] = fexp_DRAD[!, "M(Ni56)"]  
exp_corr_He[!, "Mni"] = fexp_HeS[!, "M(Ni56)"] 


function explosion_parameters(Mhe, Mco, when, reference_grid, Mmax, criterion; Ertl_calibration = "N20_old")
    #this function seeks the model in the reference grid with the closest He and CO core masses. 
    #if the chosen grid is the MW-ZAMS grid, then the maximum initial mass must be also included
    #so as to break the degeneracy between low and high-mass models that end up with the same mass
    debug =  true 
    fexp = nothing 
    ix = 0

    if reference_grid == "MW-ZAMS"
        fexp = exp_corr_S
        logM = fexp.logM
        cut = find_nearest(fexp.logM, 1.780)
        cut_range = (log10(Mmax)>1.780 ) ? ( cut:1:length(fexp.logM))  : (1:1:cut)
        print(cut_range, " ", cut )
    elseif reference_grid == "MW-Stripped"
        fexp = exp_corr_He
        cut_range = 1:1:length(fexp.E_exp)
    elseif reference_grid == "AguileraDena"
        fexp = exp_corr_AD
        cut_range = 1:1:length(fexp.E_exp)
    end

    if !isnan(Mco)
        ix = find_nearest(fexp[!, "M_co_core_"*when], Mco, seek_ini=minimum(cut_range), seek_end=maximum(cut_range))
    elseif !isnan(Mhe) 
        ix = find_nearest(fexp[!, "M_he_core_"*when], Mhe, seek_ini=minimum(cut_range), seek_end=maximum(cut_range))
    end


    if criterion == "M16" 
        debug && @printf("Eexp = %6.3e, Mco = %4.1f  (found[%3d]  %4.1f, diff = %5.2f)", fexp.E_exp[ix], Mco, ix, fexp[!, "M_co_core_"*when][ix], Mco-fexp[!, "M_co_core_"*when][ix])
        return fexp.E_exp[ix] > 0
    elseif criterion == "E16" 
        ks = Dict("S19.8"    =>[0.294, 0.0468],
        "S19.8_old"=>[0.274, 0.0470],
        "W20"      =>[0.273, 0.0413],
        "W20_old"  =>[0.284, 0.0393],
        "W15_old"  =>[0.225, 0.0495], 
        "W18_old"  =>[0.283, 0.0438], 
        "N20_old"  =>[0.194, 0.0580]
        )
        k=ks[Ertl_calibration]
        x = fexp.mu4[ix] * fexp.M4[ix]
        y = fexp.mu4[ix]
        ycrit = k[2] .* x .+ k[1]
        debug && @printf("Mco = %4.1f  (found[%3d]  %4.1f, diff = %5.2f) M4mu4=%7.3f  mu4 = %7.3f mu4_crit=%7.3f ", Mco, ix, fexp[!, "M_co_core_"*when][ix], Mco-fexp[!, "M_co_core_"*when][ix], x, y, ycrit)

        return y .< ycrit
    end

end

for Mco in 2:0.2:15
    print(explosion_parameters(NaN, Mco, "end_BURN_He", "AguileraDena", NaN, "E16"), "\n")
    
end

In [ ]:
FIGURE = Figure(size=(500,500))
ax = plot_settings(FIGURE[1, 1]; 
    xlabel = L"M_i", 
    ylabel=L"\xi", 
    title ="", xticks=0:0.1:1, yticks=0:0.1:1)
# scatter!(ax, exp_corr_AD.M, exp_corr_AD.xi25, color = "orange")
# lines!(ax, exp_corr_AD.M, exp_corr_AD.xi25, color = "gray")

ks = Dict("S19.8"    =>[0.294, 0.0468],
"S19.8_old"=>[0.274, 0.0470],
"W20"      =>[0.273, 0.0413],
"W20_old"  =>[0.284, 0.0393],
"W15_old"  =>[0.225, 0.0495], 
"W18_old"  =>[0.283, 0.0438], 
"N20_old"  =>[0.194, 0.0580]
)
k=ks["N20_old"]
x = exp_corr_AD.mu4 .* exp_corr_AD.M4
y = exp_corr_AD.mu4
print(k[1], " ", k[2])
ycrit = k[2] .* x .+ k[1]
yc(xnew) = k[2] .* xnew .+ k[1]
# scatter!(ax, exp_corr_AD.M[y .> ycrit], exp_corr_AD.xi25[y .> ycrit], color = "black")
scatter!(ax, x, y )
lines!(ax, 0:0.1:10, yc(0:0.1:10) )
xlims!(ax, 0, 1)
#ylims!(ax, 0, 0.3)
FIGURE

In [ ]:
for i in range(1, length(exp_corr_AD.M))
    @printf("%.2f %.4f %.4f\n", exp_corr_AD.M[i], exp_corr_AD.mu4[i], exp_corr_AD.M4[i])
end


In [ ]:
FIGURE = Figure(size=(500,500))
ax_tot = plot_settings(FIGURE[0:10, 0:10]; 
    xlabel = L"M_\mathrm{CO} \ [M_\odot]", 
    ylabel=L"X_\mathrm{C}", 
    title ="",
    xticks=(0:2:22.5, [L"%$(i)" for i in 0:2:22.5]), yticks=(0.:0.05:0.5, [L"%$(i)" for i in 0.:0.05:0.5]),
    ymticks=0:0.01:0.5)
ax_det = plot_settings(FIGURE[0:5, 5:10]; 
    xlabel = " ", 
    ylabel=" ", 
    title ="",
    xticks=(0:0.5:22.5, [L"%$(i)" for i in 0:0.5:22.5]), yticks=(0.:0.01:0.5, [L"%$(i)" for i in 0.:0.01:0.5]),
    ymticks=0:0.005:0.5)
    
ax_det.backgroundcolor = :white

for ax in [ax_tot, ax_det]
    ms = ax==ax_tot ? 1 : 2.5
    scatter!(ax, PS.M, PS.X, color = "lightgray", marker = :rect,  markersize=4*ms .^2, alpha = 1)
        
    MASS_LIMIT_hjin = 60.0

    filterout = 10. .^ master_f[!, "logM"].< (MASS_LIMIT_hjin)    
    filterout = 10. .^ master_f[!, "logM"] .> 0
    Xc =   (master_f[!, "Cc_1_ini_BURN_C_1"])[filterout]  *1.01
    Mc =   (master_f[!, "M_co_core_1_ini_BURN_C_1"])[filterout]
    conv = (master_f[!, "Mconv_max_C_1"])[filterout]
    colors = [ c > 0. ? :cyan : :magenta for c in conv]
    scatter!(ax, Mc,  Xc,   color=colors, marker = :utriangle, markersize = 3*ms, alpha = 0.5)

    filterout = 10. .^ master_f[!, "logM"].< (MASS_LIMIT_hjin)    
    filterout = 10. .^ master_f[!, "logM"] .> 0
    Xc =   (master_f[!, "Cc_2_ini_BURN_C_2"])[filterout]  *1.01
    Mc =   (master_f[!, "M_co_core_2_ini_BURN_C_2"])[filterout]
    conv = (master_f[!, "Mconv_max_C_2"])[filterout]
    colors = [ c > 0. ? :cyan : :magenta for c in conv]
    scatter!(ax, Mc,  Xc,   color=colors, marker = :dtriangle, markersize = 3*ms, alpha = 0.5)

    for f in [f_single_hjin_0, f_single_hjin_150, f_single_hjin_300]
        filterout = 10. .^ f[!, "logM"].< (MASS_LIMIT_hjin) 
        filterout = 10. .^ f[!, "logM"] .> 0
        Xc =   (f[!, "Cc_ini_BURN_C"])[filterout]  *1.01
        Mc =   (f[!, "M_co_core_ini_BURN_C"])[filterout]
        conv = (f[!, "Mconv_max_C"])[filterout]
        colors = [ c > 0. ? :blue : :red  for c in conv]

        scatter!(ax, Mc,  Xc,   color=colors, marker = :circle, markersize = 5*ms, alpha = 1)
    end

    for Z in keys(Zs)
        f = f_HeS_DRAD[Z]     
        Xc =   (f[!, "Cc_ini_BURN_C"])  *1.01
        Mc =   (f[!, "M_co_core_ini_BURN_C"])
        conv = (f[!, "Mconv_max_C"])
        colors = [ c > 0. ? :blue : :red for c in conv]

        scatter!(ax, Mc,  Xc,   color=colors, marker = :rect, markersize = 5*ms, alpha = Z == 0.015 ? 1 : 0.2)
    end
end

ax_det.backgroundcolor = :white
ax_tot.backgroundcolor = :white
translate!(ax_det.scene, 0, 0, 10)
translate!(ax_det.elements[:background], 0, 0, 9)

xlims!(ax_tot, 1.5,  12.5)
ylims!(ax_tot, 0.16, 0.45)
xlims!(ax_det, 5.7,  6.8)
ylims!(ax_det, 0.282, 0.307)
save("XMco_conv.pdf", FIGURE)
FIGURE


In [ ]:
x = f_single_hjin_0.M_co_core_ini_BURN_C
x[ isnan.(x)] .= +Inf
ix = find_nearest(x, 6.0)
@printf("%.3f - %.2f\n", f_single_hjin_0.logM[ix], 10. .^ f_single_hjin_0.logM[ix])



Plotting some history files to get the sense of what is happening

In [ ]:
logms = 0.5:0.1:1.8
ms = 10 .^ logms
Ms = unique([round_to_nearest_multiple(m, 2.5) for m in ms])
print(Ms)
history_files = Dict()
for m in Ms
    filename = @sprintf("%.1f.data",m)
    history_files[m] = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/drad/single_he_stars/Z_0.015/"*filename,  DataFrame, delim=' ',ignorerepeated=true)
end 
for key in names(history_files[Ms[1]])
    @printf("%20s\n", key)
end

In [ ]:
FIGURE = Figure(size=(500,500))
grid = FIGURE[1,1] = GridLayout()

ax = plot_settings(grid[5:50,55:95]; 
    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
    ylabel=L"$X_\mathrm{C}$", 
    title = " ",
    xticks=(0:0.2:1, [@sprintf("%.1f",i) for i in 0:0.2:1])
    )
#ax2 = plot_settings(grid[2, 1]; 
#    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
#    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
#    title = " ",
#    xticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1])
#)
ax3 = plot_settings(grid[0:100, 0:100]; 
    xlabel = L"M/M_i", 
    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
    title = " ",
    xticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1]),
    yticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1])
)
for m in Ms
    mod = history_files[m]
    age = mod.star_age 
    mx = mod.star_mass
    Yc = mod.center_he4
    Xc = mod.center_h1 
    Cc = mod.center_c12
    Oc = mod.center_o16
    ZAMS, TAMS, end_Hburn, ini_Heburn, end_Heburn, ini_Cburn, end_Cburn = evolutionary_checkpoints(mod, length(age), from_ZAMS=false) 
    ccm=mod.mass_conv_core
    t = (age .- age[ini_Heburn]) ./ (age[end_Heburn] - age[ini_Heburn])
    q = (mx ) ./ (mx[ini_Heburn])
    q_ccm = (ccm) ./ (mx[ini_Heburn])
    lines!(ax, t, Yc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
    lines!(ax, t, Cc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8), linestyle=:dash)
    #lines!(ax, t, Oc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8), linestyle=:dot)
    #lines!(ax2, t, q_ccm, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
    lines!(ax3, q, q_ccm, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
end

xlims!(ax, -0.02, 1)
#xlims!(ax2, -0.02, 1.1)
ylims!(ax, 0., 0.66)
#ylims!(ax2, 0., 1)
xlims!(ax3, 1., .35)
ylims!(ax3, 0.3, 0.9)

rowgap!(grid, 0)
colgap!(grid, 0)

FIGURE

In [ ]:
FIGURE = Figure(size=(500,500))
grid = FIGURE[1,1] = GridLayout()

ax = plot_settings(grid[1,1]; 
    xlabel = " ", 
    ylabel=L"$X_\mathrm{C}$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf(" ") for i in 0:0.2:2])
    )
#ax2 = plot_settings(grid[2, 1]; 
#    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
#    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
#    title = " ",
#    xticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1])
#)
ax3 = plot_settings(grid[2,1]; 
    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf("%.1f",i) for i in 0:0.2:2]),
    yticks=(0:0.1:2, [@sprintf("%.1f",i) for i in 0:0.1:2])
)
for m in Ms
    mod = history_files[m]
    age = mod.star_age 
    mx = mod.he_core_mass
    Yc = mod.center_he4
    Xc = mod.center_h1 
    Cc = mod.center_c12
    Oc = mod.center_o16
    ZAMS, TAMS, end_Hburn, ini_Heburn, end_Heburn, ini_Cburn, end_Cburn = evolutionary_checkpoints(mod, length(age), from_ZAMS=true) 
    ini_Heburn == 0  && continue
    ccm=mod.mass_conv_core
    max_conv = maximum(ccm[ini_Heburn:end_Heburn])
    @printf("m = %7.2f - conv_max = %.3f\n", m, maximum(max_conv))
    t = (age .- age[ini_Heburn]) ./ (age[end_Heburn] - age[ini_Heburn])
    q = (mx ) ./ (mx[ini_Heburn])
    q_ccm = (ccm) ./ ccm[ini_Heburn]
    lines!(ax, t, Yc, color = log10(m), colormap = :jet, colorrange=(0.9, 1.8))
    lines!(ax, t, Cc, color =log10(m), colormap = :jet, colorrange=(0.9, 1.8), linestyle=:dash)
    #lines!(ax, t, Oc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8), linestyle=:dot)
    #lines!(ax2, t, q_ccm, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
    lines!(ax3, t, ccm, color = log10(m), colormap = :jet, colorrange=(0.9, 1.8))
    lines!(ax3, [0.,1.], [max_conv, max_conv], color = log10(m), colormap = :jet, linestyle=:dash, colorrange=(0.9, 1.8))
end

xlims!(ax, -0.02, 1)
#xlims!(ax2, -0.02, 1.1)
ylims!(ax, 0., 0.66)
#ylims!(ax2, 0., 1)
xlims!(ax3, -0.02,1)
ylims!(ax3, 0.3, 71.05)

rowgap!(grid, 0)
colgap!(grid, 0)

FIGURE

In [ ]:
logms = 0.94:0.04:1.80
print(logms)
history_files = Dict()
for m in logms
    filename = @sprintf("%.3f_150.data",m)
    history_files[m] = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/single_star_grid/"*filename,  DataFrame, delim=' ',ignorerepeated=true)
end 
for key in names(history_files[logms[1]])
    @printf("%20s\n", key)
end

In [ ]:
FIGURE = Figure(size=(500,500))
grid = FIGURE[1,1] = GridLayout()

ax = plot_settings(grid[1,1]; 
    xlabel = " ", 
    ylabel=L"$X_\mathrm{C}$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf(" ") for i in 0:0.2:2])
    )
#ax2 = plot_settings(grid[2, 1]; 
#    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
#    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
#    title = " ",
#    xticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1])
#)
ax3 = plot_settings(grid[2,1]; 
    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf("%.1f",i) for i in 0:0.2:2]),
    yticks=(0.1:0.1:2, [@sprintf("%.1f",i) for i in 0.1:0.1:2])
)
for m in logms
    mod = history_files[m]
    age = mod.star_age 
    mx = mod.he_core_mass
    Yc = mod.center_he4
    Xc = mod.center_h1 
    Cc = mod.center_c12
    Oc = mod.center_o16
    ZAMS, TAMS, end_Hburn, ini_Heburn, end_Heburn, ini_Cburn, end_Cburn = evolutionary_checkpoints(mod, length(age), from_ZAMS=true) 
    ini_Heburn == 0  && continue
    ccm=mod.mass_conv_core
    max_conv = maximum(ccm[ini_Heburn:end_Heburn])
    @printf("m = %7.2f - conv_max = %.3f\n", m, maximum(max_conv))

    t = (age .- age[ini_Heburn]) ./ (age[end_Heburn] - age[ini_Heburn])
    q = (mx ) ./ (mx[ini_Heburn])
    q_ccm = (ccm) ./ (ccm[ini_Heburn])
    lines!(ax, t, Yc, color = (m), colormap = :jet, colorrange=(0.9, 1.8))
    lines!(ax, t, Cc, color = (m), colormap = :jet, colorrange=(0.9, 1.8), linestyle=:dash)
    #lines!(ax, t, Oc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8), linestyle=:dot)
    #lines!(ax2, t, q_ccm, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
    lines!(ax3, t, q_ccm, color = m, colormap = :jet, colorrange=(0.9, 1.8))
    #lines!(ax3, [0.,1.], [max_conv, max_conv], color = (m), colormap = :jet, linestyle=:dash, colorrange=(0.9, 1.8))

end

xlims!(ax, -0.02, 1)
#xlims!(ax2, -0.02, 1.1)
ylims!(ax, 0., 0.66)
#ylims!(ax2, 0., 1)
xlims!(ax3, -0.02,1)
ylims!(ax3, 0.5, 1.8)

rowgap!(grid, 0)
colgap!(grid, 0)

FIGURE

In [ ]:
logm = 1.10
logps = 0.50:0.10:3.70
q = 0.90
history_files = Dict()
for logp in logps
    filename = @sprintf("%.3f/%.3f_%.3f.data",logm, q, logp)
    history_files[logp] = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/reduced_grid/"*filename,  DataFrame, delim=' ',ignorerepeated=true)
end 
for key in names(history_files[logps[1]])
    @printf("%20s\n", key)
end

In [ ]:
FIGURE = Figure(size=(500,500))
grid = FIGURE[1,1] = GridLayout()

ax = plot_settings(grid[1,1]; 
    xlabel = " ", 
    ylabel=L"$X_\mathrm{C}$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf(" ") for i in 0:0.2:2])
    )
axc = plot_settings(grid[0,1]; 
    xlabel = L"$M_\mathrm{He-dep} [M_\odot]$", 
    ylabel=" ", 
    title = " "
    )
#ax2 = plot_settings(grid[2, 1]; 
#    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
#    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
#    title = " ",
#    xticks=(0:0.1:1, [@sprintf("%.1f",i) for i in 0:0.1:1])
#)
ax3 = plot_settings(grid[2,1]; 
    xlabel = L"$t/\Delta t_\mathrm{He-burn}$", 
    ylabel=L"$M_\mathrm{conv,He}/M_i$", 
    title = " ",
    xticks=(0:0.2:2, [@sprintf("%.1f",i) for i in 0:0.2:2]),
    yticks=(0:0.1:2, [@sprintf("%.1f",i) for i in 0:0.1:2])
)
for logp in logps
    mod = history_files[logp]
    age = mod.star_age 
    mx = mod.he_core_mass
    Yc = mod.center_he4
    Xc = mod.center_h1 
    Cc = mod.center_c12
    Oc = mod.center_o16
    ZAMS, TAMS, end_Hburn, ini_Heburn, end_Heburn, ini_Cburn, end_Cburn = evolutionary_checkpoints(mod, length(age), from_ZAMS=true) 
    ini_Heburn*end_Heburn == 0  && continue
    ccm=mod.mass_conv_core
    max_conv = maximum(ccm[ini_Heburn:end_Heburn])
    @printf("logp = %7.2f - conv_max = %.3f\n", logp, maximum(max_conv))

    t = (age .- age[ini_Heburn]) ./ (age[end_Heburn] - age[ini_Heburn])
    q = (mx ) ./ (mx[ini_Heburn])
    q_ccm = (ccm) ./ (ccm[ini_Heburn])
    lines!(ax, t, Yc, color = (logp), colormap = :jet, colorrange=(0.9, 3.8))
    lines!(ax, t, Cc, color = (logp), colormap = :jet, colorrange=(0.9, 3.8), linestyle=:dash)
    #lines!(ax, t, Oc, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8), linestyle=:dot)
    #lines!(ax2, t, q_ccm, color = log10(m), colormap = :jet, colorrange=(0.5, 1.8))
    lines!(ax3, t, q_ccm, color = logp, colormap = :jet, colorrange=(0.9, 3.8))
    lines!(axc, moving_average(t[2:end],55), moving_average(diff(Cc)./diff(t),55), color = logp, colormap = :jet, colorrange=(0.9, 3.8))
    lines!(ax3, [0.,1.], [max_conv./ (ccm[ini_Heburn]), max_conv./ (ccm[ini_Heburn])], color = (logp), colormap = :jet, linestyle=:dash, colorrange=(0.9, 3.8))

end

cmap = scatter!(ax, -logps, -logps, color=logps, colormap=:jet, colorrange=(0.9,3.8))

xlims!(ax, -0.02, 1)
#xlims!(ax2, -0.02, 1.1)
ylims!(ax, 0., 0.66)
#ylims!(ax2, 0., 1)
xlims!(axc, -0.02,1)
xlims!(ax3, -0.02,1)
ylims!(ax3, 0.5, 1.75)
ylims!(axc, -1.8, 1.5)

rowgap!(grid, 0)
colgap!(grid, 0)

Colorbar(FIGURE[1, 2], cmap, size =25, label = L"P_\mathrm{i} \ [\mathrm{d}]", 
             ticks=(logps), vertical=true, ticklabelrotation=0
)

FIGURE

In [ ]:

f_HeS_DRAD = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/drad/single_he_stars/CORR_DRAD_0.015.dat",  DataFrame, delim=' ',ignorerepeated=true)
f_single_hjin = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/CORR_GRID_single_150.dat",  DataFrame, delim=' ',ignorerepeated=true)



M_co = 9.87
grids = Dict("BG_1"=> master_f, "BG_2"=> master_f, "drad"=>f_HeS_DRAD, "hjin"=>f_single_hjin)
which = Dict("BG_1"=> 0,        "BG_2"=> 0,        "drad"=>0,                   "hjin"=>0)
history_dirs = Dict(   "drad" => "/vol/aibn133/data1/aercolino/GRID_CACHES/drad/single_he_stars/Z_0.015/",
                        "BG_1" => "/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/reduced_grid/",
                        "BG_2" => "/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/reduced_grid/",
                        "hjin"=>  "/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/single_star_grid/" 
                    )

for f in keys(grids)
    x = nothing 

    try
        x = grids[f].M_co_core_ini_BURN_C 

        try
            x[grids[f].logM .> 1.7] .= +Inf
        catch
        end
        #x = grids[f].M_co_core_end_BURN_He
    catch

        x =  occursin("1", f) ? grids[f].M_co_core_1_ini_BURN_C_1 :  grids[f].M_co_core_2_ini_BURN_C_2 
        x[grids[f].logM .> 1.7] .= +Inf

        #@printf("%s",  occursin("1", f) )
    end
    x[ isnan.(x)] .= +Inf
    ix = find_nearest(x, M_co)
    @printf("%6s %10d %7.2f  -> ", f, ix, x[ix])
    which[f] = ix
    try
        @printf("%7.2f ", grids[f].logM[ix])
        try
            @printf("%7.2f %7.2f",  grids[f].logP[ix], grids[f].q[ix])
        catch
        end
    catch
        @printf("%7.1f ", grids[f].M[ix])
    end
    @printf("\n")
end



history_file = Dict("drad" => @sprintf("%.1f.data", grids["drad"][!, "M"][which["drad"]]),
                           "BG_1" => @sprintf("%5.3f/%5.3f_%5.3f.data",   grids["BG_1"][!, "logM"][which["BG_1"]], grids["BG_1"][!, "q"][which["BG_1"]], grids["BG_1"][!, "logP"][which["BG_1"]]),
                           "BG_2" => @sprintf("%5.3f/%5.3f_%5.3f_s.data", grids["BG_2"][!, "logM"][which["BG_2"]], grids["BG_2"][!, "q"][which["BG_2"]], grids["BG_2"][!, "logP"][which["BG_2"]]),
                           "hjin"=>  @sprintf("%5.3f_150.data", grids["hjin"][!, "logM"][which["hjin"]]) )

files = Dict() 
for key in keys(grids)
    filename = history_dirs[key] * history_file[key]
    files[key] = CSV.read(filename,  DataFrame, delim=' ',ignorerepeated=true)
end

colors = Dict("BG_1"=>"blue", "BG_2"=>"green", "hjin"=>"gray", "drad"=>"red")
#print(history_file)

r3a_file = CSV.read("/vol/aibn133/data1/aercolino/test_directory/c12_ag_o16/test3a_H1.data",  DataFrame, delim=' ',ignorerepeated=true, header=["t8", "r"], skipto=5 )
rca_file = CSV.read("/vol/aibn133/data1/aercolino/test_directory/c12_ag_o16/testH1.data",  DataFrame, delim=' ',ignorerepeated=true, header=["t8", "r"], skipto=5)

r3a = interpolator(r3a_file.t8, r3a_file.r, )
rca = interpolator(rca_file.t8, rca_file.r, )
print("Done!")

In [ ]:
FIGURE = Figure(size=(600,600*sqrt(2)))
grid = FIGURE[1,1] = GridLayout()
max_rate = 0.
max_time = 0.
ax1 = plot_settings(grid[2, 1]; 
    xlabel =   " ", xticklabelsvisible=false,
    ylabel= L" M \ [M_\odot]  ", 
    title ="", xmticks=-10:0.1:10)
ax2 = plot_settings(grid[5, 1]; 
    xlabel =  L" t - t_\mathrm{He-ign} \ [\mathrm{Myr}]", 
    ylabel= L"X_\mathrm{core} ", 
    title ="", xmticks=-10:0.1:10 )
ax3 = plot_settings(grid[4, 1]; 
    xlabel =  " ", xticklabelsvisible=false,
    ylabel= L"\mathrm{d}X(\,^{12}\mathrm{C})/\mathrm{d}t",
    title ="", xmticks=-10:0.1:10 )
ax4a = plot_settings(grid[3, 1]; 
    xlabel =  " ", xticklabelsvisible=false,
    ylabel= L"T_\mathrm{c} [10^8 K]", 
    title ="", mirror_y=false, yaxisposition=:left, xmticks=-10:0.1:10 )
ax4b = plot_settings(grid[3, 1]; 
    xlabel =  " ", xticklabelsvisible=false,
    ylabel= L"\rho_\mathrm{c} \ [\mathrm{g}\,\mathrm{cm}^{-3}]", title = " ",
    mirror_y=false, yaxisposition=:right, xmticks=-10:0.1:10)
ax5 = plot_settings(grid[1, 1]; 
    xlabel =  L" t - t_\mathrm{He-ign} \ [\mathrm{Myr}]", 
    ylabel= L"\dot{M} \ [M_\odot\,\mathrm{yr}^{-1}]", 
    xaxisposition=:top, 
    title ="Comparisons between models\n with "*"M_co="*@sprintf("%.1f", M_co), titlesize=15, xmticks=-10:0.1:10)
# ax6 = plot_settings(grid[-3, 1]; 
#     xlabel =  " ", 
#     ylabel= L"M \ [M_\odot]", 
#     title ="")
print(" entering loop...\n")
for key in keys(grids) 
    print(key," ")
    occursin("2", key) ? continue : nothing 
    mod = files[key]
    

    print("retrieving data...")

    m_tot = mod.star_mass
    m_he = mod.he_core_mass
    m_co = mod.c_core_mass
    mdot = mod.log_abs_mdot
    age = mod.star_age 
    ccm = mod.mass_conv_core
    X_c = mod.center_c12 
    X_he = mod.center_he4 

    T8 = 10. .^ mod.log_center_T / 1e8
    rho = 10. .^ mod.log_center_Rho
    mu_he4 = 4.0026
    mu_c12 = 12
    rate_ca   =   1.    .* X_c .* X_he ./ (mu_c12*mu_he4) .* rho .^ 1 .* rca(T8)  
    rate_3a   =  1/6.   .* X_he .^ 3   ./ ( mu_he4)^3     .* rho .^ 2 .* r3a(T8)  
    dX_C_ca = rate_ca * mu_c12
    dX_C_3a = rate_3a * mu_c12
    dX_he   = (rate_ca + rate_3a) * mu_he4
    dX_C   = (rate_ca + rate_3a) * mu_c12
    @printf("getting evolutionary stages...")

    from_ZAMS = key=="drad" ? false : true
    ZAMS, TAMS, end_Hburn, ini_Heburn, end_Heburn, ini_Cburn, end_Cburn = evolutionary_checkpoints(mod, length(age), from_ZAMS=from_ZAMS) 
    (ZAMS ==0) && (ZAMS = 1)
    @printf("... H[%5d-%5d] He[%5d-%5d] C[%5d] end[%5d]...", ZAMS, TAMS, ini_Heburn, end_Heburn, ini_Cburn, length(age))

    #t = ( maximum(age) .- age ) ./ (age[end_Heburn]-age[ini_Heburn])
    t = age .- age[end_Heburn]
    t1 = max.(age[end_Heburn] .- age, 0.1)
    t = max.( age[ini_Cburn] .- age, 1.1)
    t_He_burn = age[end_Heburn] - age[ini_Heburn]
    t = (age .- age[ini_Heburn] ) #/ (age[end_Heburn] - age[ini_Heburn])
    t2 = age .- age[end_Heburn] 

    max_rate = max(max_rate, maximum(dX_C[ini_Heburn:end_Heburn]) )
    max_time = max(max_time, t[end_Heburn])
    print("plotting...")

    ff = ZAMS:1:ini_Cburn
    x = t/1e6
    ccm[ff[1]]=0; ccm[ff[end]]=0

    lines!(ax1, x[ff],   m_tot[ff], color = colors[key], linestyle=:solid)
    lines!(ax1, x[ff],   m_he[ff],  color = colors[key], linestyle=:dash)
    lines!(ax1, x[ff],   m_co[ff],  color = colors[key], linestyle=:dot, linewidth=2, alpha = 1)
    #lines!(ax1, x[ff],   ccm[ff],  color = colors[key], linestyle=:dot)

    #curve = [Point2f(x, y) for (x, y) in zip(log10.(t), ccm)]
    if key == "hjin"
        curve = [Point2f(x, y) for (x, y) in zip((x[ff]), ccm[ff])]
        mer_patter = Pattern("\\\\", linecolor=colors[key], background_color=:transparent, width=2, )
        poly!(ax1, curve,  color =mer_patter, strokecolor = colors[key], transparency=true,
        )
    else 
        band!(ax1,  x[ff],   ccm[ff], ccm[ff].*0,  color =  colors[key], alpha = 0.5)
    end

    lines!(ax2, x[ff],   X_c[ff],   color = colors[key], linewidth = 2, alpha = 1)
    lines!(ax2, x[ff],   X_he[ff],  color = colors[key], linestyle=:dash)
    text!(ax2, x[end_Heburn], X_c[end_Heburn], text=@sprintf("%.3f", X_c[end_Heburn]), fontsize = 8 )
    

    lines!(ax3, x[ff],  dX_C_ca[ff]/1e-12,  color = colors[key], linestyle=:solid)
    lines!(ax3, x[ff],  dX_C_3a[ff]/1e-12,  color = colors[key], linestyle=:dash)
    #lines!(ax3, x[ff],  dX_C[ff]/1e-12,     color = colors[key], linestyle=:solid)


    lines!(ax4a, x[ff],  T8[ff],  color = colors[key], linestyle=:solid)
    lines!(ax4b, x[ff],  rho[ff]./1e3,  color = colors[key], linestyle=:dash)

    #lines!(ax5, x[ff], X_he[ff] ./ dX_he[ff]/yr ./ ( age[end_Heburn] .- age[ff]), color = colors[key], linewidth = 5, alpha = 0.5, label = key)
    lines!(ax5, x[ff], mdot[ff], color = colors[key], linewidth = 2, alpha = 1, label = key)

    #lines!(ax6, x[ff], X_he[ff] .* ccm[ff], color = colors[key], linewidth = 2, alpha = 1, label = key)


    print("done!\n")
end 

if M_co == 3.00
    for ax in [ax1,ax2,ax3,ax4a,ax4b,ax5]
        xlims!(ax,-0.3e5, 8.91e5)
        xlims!(ax,-0.03, 1)
        #xlims!(ax,-0.1, 1.02)
        #xlims!(ax,1, 0)
    end

    #xlims!(ax2,1,2e4)
    ylims!(ax2, 0,0.67)
    ylims!(ax1, 0,8.8)
    ylims!(ax3, 0, max_rate*1.2/1e-12)
    ylims!(ax4a, 1., 3.75)
    ylims!(ax4b, 0.5, 3)

    ylims!(ax5, -6, -5)
elseif M_co == 1.95
    for ax in [ax1,ax2,ax3,ax4a,ax4b,ax5]
        xlims!(ax,-0.3e5, 8.91e5)
        xlims!(ax,-0.05, 1.3)
        #xlims!(ax,-0.1, 1.02)
        #xlims!(ax,1, 0)
    end

    #xlims!(ax2,1,2e4)
    ylims!(ax2, 0,0.67)
    ylims!(ax1, 0,6.2)
    ylims!(ax3, 0, max_rate*1.2/1e-12)
    ylims!(ax4a, 1., 3.75)
    ylims!(ax4b, 0.6, 4)

    ylims!(ax5, -6.7, -5)
elseif M_co == 6.2
    for ax in [ax1,ax2,ax3,ax4a,ax4b,ax5]
        xlims!(ax,-0.3e5, 8.91e5)
        xlims!(ax,-0.05, 0.6)
        #xlims!(ax,-0.1, 1.02)
        #xlims!(ax,1, 0)
    end

    #xlims!(ax2,1,2e4)
    ylims!(ax2, 0,0.67)
    ylims!(ax1, 0,19.9)
    ylims!(ax3, 0, max_rate*1.2/1e-12)
    ylims!(ax4a, 1.5, 3.5)
    ylims!(ax4b, 0.2, 2.2)

    ylims!(ax5, -5.2, -4.2)
elseif M_co == 9.87
    for ax in [ax1,ax2,ax3,ax4a,ax4b,ax5]
        xlims!(ax,-0.3e5, 8.91e5)
        xlims!(ax,-0.03, 0.42)
        #xlims!(ax,-0.1, 1.02)
        #xlims!(ax,1, 0)
    end

    #xlims!(ax2,1,2e4)
    ylims!(ax2, 0,0.67)
    ylims!(ax1, 0,29.9)
    ylims!(ax3, 0, max_rate*1.2/1e-12)
    ylims!(ax4a, 1.5, 3.5)
    ylims!(ax4b, 0.2, 2.2)

    ylims!(ax5, -5., -3.8)
else 
    print("ERROR")
end 
dump = [-1e99, -1e99]
lines!(ax1, dump,dump, color = :black, linestyle=:solid, label = "Total\nMass")
lines!(ax1, dump,dump, color = :black, linestyle=:dash, label = "He-core")
lines!(ax1, dump,dump, color = :black, linestyle=:dot, linewidth=2, alpha = 1, label = "CO-core")
curve = [Point2f(x, y) for (x, y) in zip(dump, dump)]
mer_patter = Pattern("\\\\", linecolor=:black, background_color=:gray, width=0.1, )
poly!(ax1, curve,  color =mer_patter, strokecolor = :transparent, transparency=true, label = "Conv.\nCore"
)

lines!(ax2, dump, dump,   color = :black, linewidth = 2, label = "C")
lines!(ax2, dump, dump,   color = :black, linestyle=:dash, label = "He")

lines!(ax3, dump, dump,  color = :black, linestyle=:solid, label = L"$\mathrm{C}+\alpha$")
lines!(ax3, dump, dump,  color = :black, linestyle=:dash,   label = L"$3\alpha$")

lines!(ax4a, dump,dump,  color = :black, linestyle=:solid, label = L"T")
lines!(ax4b, dump,dump,  color = :black, linestyle=:dash,  label = L"$\rho$")



axislegend(ax5,  framevisible=true, nbanks=3, halign=:center, valign=:bottom)
axislegend(ax1,  framevisible=true, nbanks=4, halign=:right, valign=:top, labelsize = 10)
axislegend(ax4a,  framevisible=true, nbanks=4, halign=:left, valign=:top, labelsize = 10)
axislegend(ax4b,  framevisible=true, nbanks=4, halign=:right, valign=:bottom, labelsize = 15)
axislegend(ax3,  framevisible=true, nbanks=2, halign=:center, valign=:top, labelsize = 15)
axislegend(ax2,  framevisible=true, nbanks=2, halign=:left, valign=:top, labelsize = 15)

rowgap!(grid, 0)
#colgap!(grid, 0)
savedir = "/vol/aibn133/data1/aercolino/GRID_CACHES/"*@sprintf("Mco%.2f.pdf", M_co)
save(savedir, FIGURE)

FIGURE

In [ ]:
f24=CSV.read("/vol/aibn133/data1/aercolino/CATALOGUES/Schneider2024/table_models_zenodo.csv", DataFrame)
deleteat!(f24, findall( ismissing , f24[!, " t_cc/Myr"]) )


FIG = Figure(size=(600,600))


ax = plot_settings(FIG[1,1]; 
    xlabel =   " ", 
    ylabel=  "    ", 
    title ="", )
# to_ignore = .! (occursin.("A", f24[!, " Case"]) .|| occursin.("C", f24[!, " Case"]) )
to_plot =   " Case B-e" .== f24[!, " Case"]
Mini = convert.(Float64,  f24[!, "M_ini/Msun"])
Mend = convert.(Float64,  f24[!, " M_final/Msun"])
Mco = convert.(Float64,  f24[!, " M_CO/Msun"])
f_acc = [k == " " ? 0. : parse(Float64, k)  for k in f24[!, " f_acc"]] 


singles = ( " Single" .== f24[!, " Case"])
Mco_single  = interpolator(Mini[singles], Mco[singles],   extrapolation=:flat)
Mend_single = interpolator(Mini[singles], Mend[singles],  extrapolation=:flat)

Mend_frac =   Mend  ./ Mend_single(Mini)
Mend_frac_new =   Mend  ./ Mend_single(Mini .* (1 .+ f_acc) )
# print(length(Mend[to_plot]), " ", length(Mini[to_plot]))
print(f_acc)

# cscat = scatter!(ax, Mini[to_plot], Mend_frac[to_plot],  color = f_acc[to_plot],
#  colormap=:jet, colorrange =[0,2], marker = :rect)
#  cscat = scatter!(ax, Mini[to_plot], Mend_frac_new[to_plot],  color = f_acc[to_plot],
#  colormap=:jet, colorrange =[0,2], marker = :cross)
lines!(Mini[singles], Mend_single(Mini[singles]) ./Mend_single(Mini[singles]))
lines!(Mini[singles], Mend_single(Mini[singles]*(1+0.1)) ./Mend_single(Mini[singles]),colormap=:jet, color = 0.10, colorrange =[0,2])
lines!(Mini[singles], Mend_single(Mini[singles]*(1+0.25)) ./Mend_single(Mini[singles]),colormap=:jet, color = 0.25, colorrange =[0,2])
lines!(Mini[singles], Mend_single(Mini[singles]*(1+0.50)) ./Mend_single(Mini[singles]),colormap=:jet, color = 0.50, colorrange =[0,2])
lines!(Mini[singles], Mend_single(Mini[singles]*(1+0.75)) ./Mend_single(Mini[singles]),colormap=:jet, color = 0.75, colorrange =[0,2])
# lines!(Mini[singles], Mend_single(Mini[singles]*(1+1)),colormap=:jet, color = 1, colorrange =[0,2])
# lines!(Mini[singles], Mend_single(Mini[singles]*(1+1.5)),colormap=:jet, color = 1.5, colorrange =[0,2])
# lines!(Mini[singles], Mend_single(Mini[singles]*(1+2)),colormap=:jet, color =2, colorrange =[0,2])
scatter!(Mini[to_plot], Mend[to_plot] ./Mend_single(Mini[to_plot]), color = f_acc[to_plot],
   colormap=:jet, colorrange =[0,2], marker = :cross)
 
# Colorbar(FIG[1, 2], cscat, size =25, label = L"f_\mathrm{acc}", 
#              vertical=true, ticklabelrotation=0)
# ylims!(ax,1,100)
# xlims!(ax,10,75)
FIG

In [ ]:
f24=CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/Table_A.1_amended.csv", DataFrame)
deleteat!(f24, findall( ismissing , f24[!, " t_cc/Myr"]) )
using Makie.Colors
colors = [RGBAf(0.1, 0.0, 0.1, 0.3), 
          RGBAf(1, 0., 1, 0.3), # blue
          RGBAf(1, .8, 1, .5),  # yellow
          RGBAf(1, 1, 1, 1),  # yellow
          RGBAf(.8, 1, .8, .5),  # yellow
          RGBAf(0., 1, 0., 0.3),
          RGBAf(0. , 0.1, 0. , 0.3)]  # red
positions = [0.0, 0.25, 0.42, 0.5, 0.58, 0.75, 1.0]

my_cmap = cgrad(colors, positions, categorical = false)


FIG = Figure(size=(600,500))


ax = plot_settings(FIG[1,1]; 
    xlabel =   L"M_\mathrm{ZAMS}", 
    ylabel=  L"f_\mathrm{acc}   =   \Delta M/M_\mathrm{ZAMS}", 
    title ="",  xticks = [10,20,30,40,50,60,70], xmticks=10:1:80, xscale = log10,
    yticks=[0.1, 0.25, 0.5, 0.75, 1, 1.5, 2])
# to_ignore = .! (occursin.("A", f24[!, " Case"]) .|| occursin.("C", f24[!, " Case"]) )
Mini = f24[!, "M_ini/Msun"]
# Mend = f24[!, "M_ini/Msun"] .* (1 .+ parse.(Float64, f24[!, " f_acc"]))
Mco = f24[!, " M_CO/Msun"]
f_acc = [k == " " ? 0 : parse(Float64, k)  for k in f24[!, " f_acc"]] 
singles = ( " Single" .== f24[!, " Case"])
Mco_single   = interpolator(Mini[singles], Mco[singles],   extrapolation=:flat)

to_plot =    " Case B-l" .== f24[!, " Case"] 








# cmap = Reverse(:vanimo)
cmap = :corkO
cmap = my_cmap
crange = [0.65, 1.35]
x = Mini[to_plot]
y = convert.(Float64, f_acc[to_plot])
zc = Mco[to_plot]./Mco_single(Mini[to_plot])
# heatmap!(ax, x, y, zc, colormap=cmap, colorrange =[0.65,1.35], 
#             interpolate=false, alpha = 0.5, nan_color = :gray)
tricontourf!(ax, x, y, zc, colormap = cmap, transparency=true,  levels=0.65:0.025:1.35)

logT_eff_CC = f24[!, " log Teff_cc/K"]

bsg_col = [logt>3.9 ? :blue : :red for logt in logT_eff_CC ]
BSG = [logt>3.9 ? 1 : -1 for logt in logT_eff_CC ]
Teff = convert.(Float64, logT_eff_CC[to_plot])
z2 = Teff
cscat = scatter!(ax,  x, y, color =  zc,
 colormap=cmap, colorrange =crange, marker = :rect, strokecolor=bsg_col[to_plot], strokewidth = 2, alpha = 1)
# fac_mid1 = (0.50+0.75)/2
# fac_mid2 = (0.75+1)/2
# fac_mid0 = (0.50+0.25)/2
# BOUNDS_RSG_xy = [(10,fac_mid1), (16, fac_mid1), (16,fac_mid0), (22, fac_mid0), 
#                  (22,fac_mid1), (38,fac_mid1), (38,fac_mid2), (45,fac_mid2),
#                  (45,0)]
# lines!(ax, BOUNDS_RSG_xy, color=:blue) 
contour!(ax, x, y, BSG[to_plot], levels = [0])

Colorbar(FIG[1, 2], cscat, size =25, label = L"M_\mathrm{CO,f}/M_\mathrm{CO}(M_\mathrm{i})", 
             vertical=true, ticklabelrotation=0, ticks=0:.1:2)
ylims!(ax,0.0,2.1)
xlims!(ax,10,75)
FIG

In [ ]:
f24

In [ ]:
# for i in range(1,length(x))
#     x[i] < 27 && continue 
#     @printf("%.3f %.3f %.3f\n", x[i], y[i], Teff[i])
# end

x_unique = sort(unique(x))
y_unique = sort(unique(y))
nx = length(x_unique)
ny = length(y_unique)
A = fill(NaN, nx, ny)  # fill missing values with NaN for now
x_idx = Dict(val => i for (i, val) in enumerate(x_unique))
y_idx = Dict(val => i for (i, val) in enumerate(y_unique))
for (xi, yi, zi) in zip(x, y, z2)
    A[x_idx[xi], y_idx[yi]] = zi
end


function fill_nan_linear_x(A)
    A_filled = copy(A)
    for j in 1:size(A, 2)
        for i in 1:size(A, 1)
            if isnan(A[i, j])
                # Search for neighbors in x-direction (i-axis)
                left = findlast(!isnan, A[1:i-1, j])
                right = findfirst(!isnan, A[i+1:end, j])
                right = isnothing(right) ? nothing : right + i  # adjust index if found

                if !isnothing(left) && !isnothing(right)
                    # Linear interpolation
                    x1, x2 = x_unique[left], x_unique[right]
                    z1, z2 = A[left, j], A[right, j]
                    x_missing = x_unique[i]
                    A_filled[i, j] = z1 + (z2 - z1) * (x_missing - x1) / (x2 - x1)
                elseif !isnothing(left)
                    A_filled[i, j] = A[left, j]  # Nearest on the left
                elseif !isnothing(right)
                    A_filled[i, j] = A[right, j]  # Nearest on the right
                end
            end
        end
    end
    return A_filled
end

A_filled = fill_nan_linear_x(A)

# Step 4: Create interpolator
nodes = (x_unique, y_unique)
mf_to_teff =  extrapolate(interpolate(nodes, A_filled, Gridded(Linear())), Flat())



In [ ]:
include("/vol/aibn133/data1/aercolino/SOFTWARE/Julia/SN_support.jl")
mf_to_teff_l

In [ ]:
# include("/vol/aibn133/data1/aercolino/SOFTWARE/Julia/SN_support.jl")



f24=CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/Table_A.1_amended.csv", DataFrame)
deleteat!(f24, findall( ismissing , f24[!, " t_cc/Myr"]) )


FIG = Figure(size=(600,500))


ax = plot_settings(FIG[1,1]; 
    xlabel =   L"M_\mathrm{ZAMS}", 
    ylabel=  L"f_\mathrm{acc}   =   \Delta M/M_\mathrm{ZAMS}", 
    title ="",  xticks = [10,20,30,40,50,60,70], xmticks=10:1:80, xscale = log10,
    yticks=[0.1, 0.25, 0.5, 0.75, 1, 1.5, 2])
# to_ignore = .! (occursin.("A", f24[!, " Case"]) .|| occursin.("C", f24[!, " Case"]) )
Mini = f24[!, "M_ini/Msun"]
# Mend = f24[!, "M_ini/Msun"] .* (1 .+ parse.(Float64, f24[!, " f_acc"]))
Mco = f24[!, " M_CO/Msun"]
f_acc = [k == " " ? 0 : parse(Float64, k)  for k in f24[!, " f_acc"]] 
singles = ( " Single" .== f24[!, " Case"])
Mco_single   = interpolator(Mini[singles], Mco[singles],   extrapolation=:flat)

to_plot =    " Case B-l" .== f24[!, " Case"] 








# cmap = Reverse(:vanimo)
cmap = Reverse(:balance)
crange = [3.45, 4.35]
x = Mini[to_plot]
y = convert.(Float64, f_acc[to_plot])
zc = Mco[to_plot]./Mco_single(Mini[to_plot])

bsg_col = [logt>3.9 ? :blue : :red for logt in logT_eff_CC ]
BSG = [logt>3.9 ? 1 : -1 for logt in logT_eff_CC ]
Teff = convert.(Float64, logT_eff_CC[to_plot])
z2 = Teff

# heatmap!(ax, x, y, zc, colormap=cmap, colorrange =[0.65,1.35], 
#             interpolate=false, alpha = 0.5, nan_color = :gray)
tricontourf!(ax, x, y, z2, colormap = cmap, transparency=true,  
            levels=3.45:0.025:4.35, extendhigh=:auto)

logT_eff_CC = f24[!, " log Teff_cc/K"]
cscat = scatter!(ax,  x, y, color =  z2,
 colormap=cmap, colorrange =crange, marker = :rect, 
 strokecolor=:black, strokewidth = 0.0, alpha = 1, highclip=:blue)
# fac_mid1 = (0.50+0.75)/2
# fac_mid2 = (0.75+1)/2
# fac_mid0 = (0.50+0.25)/2
# BOUNDS_RSG_xy = [(10,fac_mid1), (16, fac_mid1), (16,fac_mid0), (22, fac_mid0), 
#                  (22,fac_mid1), (38,fac_mid1), (38,fac_mid2), (45,fac_mid2),
#                  (45,0)]
# lines!(ax, BOUNDS_RSG_xy, color=:blue) 
for _logx in 1:0.01:2
    _x = 10^_logx
    for _y in 0.1:0.02:1.5
        #print(mf_to_teff(_x,_y), "\n")
        _z = mf_to_teff_l(_x,_y)
        3.85 < _z < 3.95 &&  scatter!(ax, _x, _y, color=:black,
             marker = '.', markersize=25,
            alpha = 0.2, highclip=:blue)
       
    end
end
Colorbar(FIG[1, 2], cscat, size =25, label = L"M_\mathrm{CO,f}/M_\mathrm{CO}(M_\mathrm{i})", 
             vertical=true, ticklabelrotation=0, ticks=3.4:.1:5.1)
ylims!(ax,0.0,2.1)
xlims!(ax,10,75)
FIG

In [ ]:
Mi = 9
f_acc = 0.5

Mco_end_i = interpol_single_Mmax["M_cocore_end"](Mi)
Mco_boost = mf_to_mco_e(Mi,f_acc)
BSG = mf_to_teff_e(Mi,f_acc) > 3.9
@printf("Mi = %.2f, Mco,f = %.2f, boost=%.2f -> Mf = %.2f, Mco = %.2f, [%s]",
        Mi, Mco_end_i,f_acc, Mi*(1+f_acc), Mco_end_i*Mco_boost, BSG ? "BSG" : "RSG" )

In [ ]:
f24=CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/Table_A.1_amended.csv", DataFrame)
deleteat!(f24, findall( ismissing , f24[!, " t_cc/Myr"]) )


FIG = Figure(size=(600,500))
using Makie.Colors
colors = [RGBAf(0.1, 0.0, 0.1, 0.3), 
          RGBAf(1, 0., 1, 0.3), # blue
          RGBAf(1, .8, 1, .5),  # yellow
          RGBAf(1, 1, 1, 1),  # yellow
          RGBAf(.8, 1, .8, .5),  # yellow
          RGBAf(0., 1, 0., 0.3),
          RGBAf(0. , 0.1, 0. , 0.3)]  # red
positions = [0.0, 0.25, 0.42, 0.5, 0.58, 0.75, 1.0]

my_cmap = cgrad(colors, positions, categorical = false)


ax = plot_settings(FIG[1,1]; 
    xlabel =   L"M_\mathrm{ZAMS}", 
    ylabel=  L"f_\mathrm{acc}   =   \Delta M/M_\mathrm{ZAMS}", 
    title ="",  xticks = [10,20,30,40,50,60,70], xmticks=10:1:80, xscale = log10,
    yticks=[0.1, 0.25, 0.5, 0.75, 1, 1.5, 2])
# to_ignore = .! (occursin.("A", f24[!, " Case"]) .|| occursin.("C", f24[!, " Case"]) )
Mini = f24[!, "M_ini/Msun"]
# Mend = f24[!, "M_ini/Msun"] .* (1 .+ parse.(Float64, f24[!, " f_acc"]))
Mco = f24[!, " M_CO/Msun"]
f_acc = [k == " " ? 0 : parse(Float64, k)  for k in f24[!, " f_acc"]] 
singles = ( " Single" .== f24[!, " Case"])
Mco_single   = interpolator(Mini[singles], Mco[singles],   extrapolation=:flat)

to_plot =    " Case B-l" .== f24[!, " Case"] 








# cmap = Reverse(:vanimo)
cmap = :corkO
cmap = my_cmap
crange = [0.65, 1.35]
x = Mini[to_plot]
y = convert.(Float64, f_acc[to_plot])
zc = Mco[to_plot]./Mco_single(Mini[to_plot])
# heatmap!(ax, x, y, zc, colormap=cmap, colorrange =[0.65,1.35], 
#             interpolate=false, alpha = 0.5, nan_color = :gray)
tricontourf!(ax, x, y, zc, colormap = cmap, transparency=true,  levels=0.65:0.025:1.35)

logT_eff_CC = f24[!, " log Teff_cc/K"]

bsg_col = [logt>3.9 ? :blue : :red for logt in logT_eff_CC ]
BSG = [logt>3.9 ? 1 : -1 for logt in logT_eff_CC ]
Teff = convert.(Float64, logT_eff_CC[to_plot])
z2 = Teff
cscat = scatter!(ax,  x, y, color =  zc,
 colormap=cmap, colorrange =crange, marker = :rect, strokecolor=bsg_col[to_plot], strokewidth = 2, alpha = 1)

 
for _logx in 1:0.005:2
    _x = 10^_logx
    for _y in 0.0:0.01:1.
        #print(mf_to_teff(_x,_y), "\n")
        _z = mf_to_teff(_x,_y)
        3.88 < _z < 3.92 &&  scatter!(ax, _x, _y, color=:black,
             marker = '.', markersize=15,
            alpha = 0.3, highclip=:blue)
       
    end
end

Colorbar(FIG[1, 2], cscat, size =25, label = L"M_\mathrm{CO,f}/M_\mathrm{CO}(M_\mathrm{i})", 
             vertical=true, ticklabelrotation=0, ticks=0:.1:2)
ylims!(ax,0.0,2.1)
xlims!(ax,10,75)
FIG

In [ ]:
include("SN_reader_MW.jl")
MODELS = predict_SN_from_each_model(
    "PABLO","TOTAL","X", "MM", Inf
)
using CairoMakie

In [ ]:
f24=CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/Table_A.1_amended.csv", DataFrame)
deleteat!(f24, findall( ismissing , f24[!, " t_cc/Myr"]) )


FIG = Figure(size=(400,600))

grid = FIG[1,1] = GridLayout()
xticks_v=union(1:1:10, 10:10:100)
yticks_v=[.1,.25,.5,.75,1]
xticks_t = [L"%$(i)" for i in xticks_v]
yticks_t = [L"%$(@sprintf(\"%.2f\",i))" for i in yticks_v]
xticks=(xticks_v,xticks_t)
yticks=(yticks_v,yticks_t)

ax = plot_settings(grid[1,1]; 
    xlabel =    " ", 
    ylabel=  L"f_\mathrm{acc}   =   \Delta M/M_\mathrm{ZAMS}", 
    title ="", 
    yticklabelsize=14,  
    ylabelsize=18,
    xticklabelsize=14,  
    xlabelsize=18,
    xticks = xticks, xmticks=1:1:80, xscale = log10,
    yticks=yticks,xticklabelsvisible=false)
ax2 = plot_settings(grid[2,1]; 
    xlabel =   L"M_\mathrm{ZAMS}", 
    yticklabelsize=14,  
    ylabelsize=18,
    xticklabelsize=14,  
    xlabelsize=18,
    ylabel=  L"f_\mathrm{acc}   =   \Delta M/M_\mathrm{ZAMS}", 
    title ="",  xticks = xticks, xmticks=1:1:80, xscale = log10,
    yticks=yticks)

Mini = f24[!, "M_ini/Msun"]
Mco = f24[!, " M_CO/Msun"]
f_acc = [k == " " ? 0 : parse(Float64, k)  for k in f24[!, " f_acc"]] 
singles = ( " Single" .== f24[!, " Case"])
Mco_single   = interpolator(Mini[singles], Mco[singles],   extrapolation=:flat)

using Makie.Colors
colors = [RGBAf(0.2, 0.1, 0., 0.3), 
          RGBAf(1, 1., 0, 0.3), # orange
          RGBAf(1, 1, .8, .5),  #  
          RGBAf(1, 1, 1, 1),  #  
          RGBAf(.8, 1, .8, .5),  #  
          RGBAf(0., 1, 0., 0.3),
          RGBAf(0. , 0.1, 0. , 0.3)]  # green
positions = [0.0, 0.25, 0.42, 0.5, 0.58, 0.75, 1.0]
my_cmap = cgrad(colors, positions, categorical = false)

early_or_late = "early"

f24_filter = early_or_late == "early" ? "e" : "l"
to_plot =  " Case B-"*f24_filter .== f24[!, " Case"] 
mf_to_mco  =  early_or_late == "early" ? mf_to_mco_e : mf_to_mco_l
mf_to_teff = early_or_late == "early" ? mf_to_teff_e : mf_to_teff_l
mf_to_mhe  = early_or_late == "early" ? mf_to_mhe_e : mf_to_mhe_l


# cmap = Reverse(:vanimo)
cmap = my_cmap
crange = [0.65, 1.35]
x = Mini[to_plot]
y = convert.(Float64, f_acc[to_plot])
zc = Mco[to_plot]./Mco_single(Mini[to_plot])
tricontourf!(ax, x, y, zc, colormap = cmap, transparency=true,  levels=0.65:0.025:1.35)
logT_eff_CC = f24[!, " log Teff_cc/K"]

bsg_col = [logt>3.9 ? :blue : :red for logt in logT_eff_CC ]
BSG = [logt>3.9 ? 1 : -1 for logt in logT_eff_CC ]
Teff = convert.(Float64, logT_eff_CC[to_plot])
cscat = scatter!(ax,  x, y, color =  zc,
  colormap=cmap, colorrange =crange, marker = :rect, strokecolor=bsg_col[to_plot], strokewidth = 2, alpha = 1)

 
for _logx in 0.5:0.005:2
    _x = 10^_logx
    for _y in 0.0:0.005:1.3
        #print(mf_to_teff(_x,_y), "\n")
        _z = mf_to_teff(_x,_y)
        3.89 <= _z <= 3.91 &&  scatter!(ax, _x, _y, color=:pink,
             marker = '.', markersize=15,
            alpha = 1 )
        3.88 < _z < 3.89 &&  scatter!(ax, _x, _y, color=:red,
            marker = '.', markersize=15,
           alpha = 1 )
        3.91 < _z < 3.92 &&  scatter!(ax, _x, _y, color=:blue,
           marker = '.', markersize=15,
          alpha = 1 )

    end
end


for logM in logMs
    print()
    logM_t = to_key(logM)  
    for logP in logPs 
        logP_t = to_key(logP)
        for q in qs 
            q_t = to_key(q)
            Mi = 10^logM
            facc = -1
            try 
                facc=MODELS[logM_t][logP_t][q_t]["facc_B"]
            catch 
                continue 
            end 
            mod = MODELS[logM_t][logP_t][q_t]
            Mco_end =  interpol_single_Mmax["M_cocore_end"](Mi)*mf_to_mco(Mi, facc)
            Mhe_end =  interpol_single_Mmax["M_hecore_end"](Mi)*mf_to_mhe(Mi, facc)
            model = mod["mod"]
            (isnan(facc) || occursin("A", mod["case"]) || ! occursin(early_or_late, mod["casus"] )) && continue
            BSG = mf_to_teff(Mi, facc) > 3.9
            SN = (Mco_end >= 1.44 || Mhe_end >= 2.49)
            explode = SN ? will_it_explode(Mhe_end, Mco_end+0.5, NaN, Mi, "end", "MM", "MW-ZAMS"; debug=false) : SN
            color = !explode ? "lightgray" : "black"
            scatter!(ax,  Mi, facc, color = color, markersize=25, marker = '-')
            (facc > 1) &&  print("$model ", logM_t, " ", logP_t, " ", q_t, ": ", facc, "\n")

        end
    end
end


early_or_late = "late"

f24_filter = early_or_late == "early" ? "e" : "l"
to_plot =  " Case B-"*f24_filter .== f24[!, " Case"] 
mf_to_mco  =  early_or_late == "early" ? mf_to_mco_e : mf_to_mco_l
mf_to_teff = early_or_late == "early" ? mf_to_teff_e : mf_to_teff_l
mf_to_mhe  = early_or_late == "early" ? mf_to_mhe_e : mf_to_mhe_l


# cmap = Reverse(:vanimo)
cmap = my_cmap
crange = [0.65, 1.35]
x = Mini[to_plot]
y = convert.(Float64, f_acc[to_plot])
zc = Mco[to_plot]./Mco_single(Mini[to_plot])
tricontourf!(ax2, x, y, zc, colormap = cmap, transparency=true,  levels=0.65:0.025:1.35)
logT_eff_CC = f24[!, " log Teff_cc/K"]

bsg_col = [logt>3.9 ? :blue : :red for logt in logT_eff_CC ]
BSG = [logt>3.9 ? 1 : -1 for logt in logT_eff_CC ]
Teff = convert.(Float64, logT_eff_CC[to_plot])
cscat = scatter!(ax2,  x, y, color =  zc,
  colormap=cmap, colorrange =crange, marker = :rect, strokecolor=bsg_col[to_plot], strokewidth = 2, alpha = 1)

 
for _logx in 0.5:0.005:2
    _x = 10^_logx
    for _y in 0.0:0.005:1.3
        #print(mf_to_teff(_x,_y), "\n")
        _z = mf_to_teff(_x,_y)
        3.89 <= _z <= 3.91 &&  scatter!(ax2, _x, _y, color=:pink,
             marker = '.', markersize=15,
            alpha = 1 )
        3.88 < _z < 3.89 &&  scatter!(ax2, _x, _y, color=:red,
            marker = '.', markersize=15,
           alpha = 1 )
        3.91 < _z < 3.92 &&  scatter!(ax2, _x, _y, color=:blue,
           marker = '.', markersize=15,
          alpha = 1 )

    end
end


for logM in logMs
    print()
    logM_t = to_key(logM)  
    for logP in logPs 
        logP_t = to_key(logP)
        for q in qs 
            q_t = to_key(q)
            Mi = 10^logM
            facc = -1
            try 
                facc=MODELS[logM_t][logP_t][q_t]["facc_B"]
            catch 
                continue 
            end 
            mod = MODELS[logM_t][logP_t][q_t]
            Mco_end =  interpol_single_Mmax["M_cocore_end"](Mi)*mf_to_mco(Mi, facc)
            Mhe_end =  interpol_single_Mmax["M_hecore_end"](Mi)*mf_to_mhe(Mi, facc)
            model = mod["mod"]
            (isnan(facc) || occursin("A", mod["case"]) || ! occursin(early_or_late, mod["casus"] )) && continue
            BSG = mf_to_teff(Mi, facc) > 3.9
            SN = (Mco_end >= 1.44 || Mhe_end >= 2.49)
            explode = SN ? will_it_explode(Mhe_end, Mco_end+0.5, NaN, Mi, "end", "MM", "MW-ZAMS"; debug=false) : SN
            color = !explode ? "lightgray" : "black"
            scatter!(ax2,  Mi, facc, color = color, markersize=25, marker = '-')
            (facc > 1) &&  print("$model ", logM_t, " ", logP_t, " ", q_t, ": ", facc, "\n")

        end
    end
end

ticks=(0:.1:2, [L"%$(i)" for i in 0.:.1:2])
Colorbar(grid[0,1], cscat, size =25, label = L"M_\mathrm{CO,f}/M_\mathrm{CO}(M_\mathrm{i})", 
              vertical=false, ticklabelrotation=0, ticks=ticks,
              labelsize=18, ticklabelsize=14)
ylims!(ax,0.06,1.04)
xlims!(ax,5,50)
ylims!(ax2,0.06,1.04)
xlims!(ax2,5,50)
                          
rowgap!(grid, 1, 15)
rowgap!(grid, 2, 5)
# colgap!(grid, 15)

FIG

In [ ]:
print_history(f_MW["0.90"], 180, ["M_1", "M_he_core_1", "M_2", "M_he_core_2"], "2")
MODELS["0.90"]["0.45"]["0.95"]
interpol_single_Mmax["M_cocore_end"](10)
f24 
8

In [ ]:
include("SN_reader_MW.jl")
MODELS = predict_SN_from_each_model(
    "X","TOTAL","X", "X", Inf
)
using CairoMakie

In [ ]:
FIG = Figure(size=(400,400))

grid = FIG[1,1] = GridLayout()
xticks_v=union(1:1:10, 10:10:100)
yticks_v=[.1,.25,.5,.75,1]
xticks_t = [L"%$(i)" for i in xticks_v]
yticks_t = [L"%$(@sprintf(\"%.2f\",i))" for i in yticks_v]
xticks=(xticks_v,xticks_t)
yticks=(yticks_v,yticks_t)

ax = plot_settings(grid[1,1]; 
    xlabel =    L"M_\mathrm{core} \ [M_\odot]", 
    ylabel=  L"M_\mathrm{env}   \ [M_\odot]", 
    title ="", 
    yticklabelsize=14,  
    ylabelsize=18,
    xticklabelsize=14,  
    xlabelsize=18,
    xticks=([0.3,1,3,10,30]), xmticks=union(0.1:0.1:1,1:1:10,10:10:100),
    xscale=log10,yscale=log10,
    yticks=([0.3,1,3,10,20]), ymticks=union(0.1:0.1:1,1:1:10,10:10:100))
# axM = plot_settings(grid[1,2]; 
#     xlabel =    L"M_\mathrm{core} \ [M_\odot]", 
#     ylabel=  L"M_\mathrm{env}   \ [M_\odot]", 
#     title ="", 
#     yticklabelsize=14,  
#     ylabelsize=18,
#     xticklabelsize=14,  
#     xlabelsize=18,
#     xticks=([0.3,1,3,10,30]), xmticks=union(0.1:0.1:1,1:1:10,10:10:100),
#     xscale=log10,yscale=log10,
#     yticks=([0.3,1,3,10,20]), ymticks=union(0.1:0.1:1,1:1:10,10:10:100))

fp = open("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/Mcore_Menv_corr_new.dat", "w")
write(fp, @sprintf("%15s %15s %5s %5s %5s %5s %15s %15s %15s %7s\n", "Mhe", "Menv", "WD", "II", "IIb", "Ibc", "Mend", "Mhecore_end", "Mcocore_end", "Xc"))

for logM in 0.95:0.05:1.80
    logM_t = to_key(logM)  
    f = f_MW[logM_t]

    for logP in logPs 
        logP_t = to_key(logP)
        for q in qs 
            q_t = to_key(q)
            Mi = 10^logM
            facc = -1
            try 
                facc=MODELS[logM_t][logP_t][q_t]
            catch 
                continue 
            end 
            mod = MODELS[logM_t][logP_t][q_t]
            model = mod["mod"]
            occursin("mer", mod["casus"]) && continue 
            f[!, "deltaM_A"][model] > 0 && continue 
            !(f[!, "M_he_core_1_end_BURN_He_1"][model] > 0) && continue 

            # (isnan(facc) || occursin("A", mod["case"]) || ! occursin(early_or_late, mod["casus"] )) && continue
            color = :black
            SN_primary = (mod["first"]=="1") ? mod["SN1"] : mod["SN2"] 
            (SN_primary["WD"]  > 0) && (color = "gray")
            (SN_primary["IIP"] > 0) && (color = "red")
            (SN_primary["Ibc"]+SN_primary["Ibc-i"] > 0) && (color = "blue")
            (SN_primary["IIb"] > 0) && (color = "green")
            (SN_primary["II-i"] > 0) && (color = "red")
            (SN_primary["X"] > 0) && continue
            
            Mhe = f[!, "M_he_core_1_ini_BURN_He_1"][model] 
            Menv = max(f[!, "M_1_ini_BURN_He_1"][model]-Mhe, 0.001)
            scatter!(ax,   Mhe, Menv, color = color, markersize=8, marker = :rect)
            # (facc > 1) &&  print("$model ", logM_t, " ", logP_t, " ", q_t, ": ", facc, "\n")
            Mend = f[!,"M_1_end_BURN_He_1"][model] 
            Mhecore_end = f[!, "M_he_core_1_end_BURN_He_1"][model]
            Mcocore_end = f[!, "M_co_core_1_end_BURN_He_1"][model]
            Xc = f[!, "Cc_1_end_BURN_He_1"][model]

            # scatter!(axM,  Mhe, Menv, color = Xc, colorrange=[0.2,0.5], colormap=:jet, markersize=8, marker = :rect)
         
 
            write(fp, @sprintf("%15.3f %15.3f %5d %5d %5d %5d %15.3f %15.3f %15.3f %7.3f\n", Mhe, Menv,
            SN_primary["WD"], SN_primary["IIP"]+SN_primary["II-i"], SN_primary["IIb"], SN_primary["Ibc"]+SN_primary["Ibc-i"],
             Mend, Mhecore_end, Mcocore_end, Xc))

    
    


        end
    end
end

d12 = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/Mcore_Menv_corr_new.dat",
 DataFrame,delim=' ', ignorerepeated=true)
Mhe=d12.Mhe
Menv=d12.Menv

logM_core = log10.(Mhe)
logM_env  = log10.(Menv)
for logm_core in 0:0.025:1.5
    for logm_env in -1:0.025:1.5 

        ix = get_nearest_category_weighted_ix(logM_core,logM_env, logm_core, logm_env; yweight=1/5)

        color = "black"
        d12.WD[ix]  > 0 && (color = "gray")
        d12.II[ix]  > 0 && (color = "red")
        d12.Ibc[ix]  > 0 && (color = "blue")
        d12.IIb[ix]  > 0 && (color = "green")
        
        scatter!(ax,   10^logm_core, 10^logm_env, color = color, markersize=5, marker = :rect, alpha = 0.5)

    end
end

xlims!(ax, 1.5, 30)
ylims!(ax, 0.4, 20)
close(fp)

# xlims!(axM, 1., 30)
# ylims!(axM, 0.4, 20)


FIG

# Generate Menv_Mcore correlation data

In [ ]:
using Revise 
using POPSYNTH
using Printf 
using StatsBase 
using CSV 

include("/vol/aibn133/data1/aercolino/SOFTWARE/SN_from_grids/JuliaScripts/Plot_defs.jl")
f_MW = POPSYNTH.read_hdf5_into_dataframe(POPSYNTH.BG_dir*"master_CORR.hdf5")

logMs = range(0.70, 2.001, step=0.050)
logPs = range(0.05, 3.70, step=0.050)
qs    = range(0.10, 0.951, step=0.050)


MERGER_CRITERION = "X"  #  X, PA_IV, ERK, PABLO, PAULI
MERGER_MASSLOSS  = "X"  #"FIX"+fraction or "Energy"
MERGER_EOL       = "CORE"  #"TOTAL" or "CORE" or "ANALYTIC"
EXP_CRIT =  "X" # "MM-S21"  #"Ertl-N20_new" #  "PS20"
KICKS = "Inf"  

f_B = 0.75
IMF_exponent = POPSYNTH.alpha_m 
P_exponent   = 0                 #alpha_p-stdev_p
q_exponent   = 0                 #alpha_q+stdev_q

@printf("analyzing grid...\n")
MODELS = nothing 
SN_DATA = nothing 
GC.gc()

MODELS  = predict_SN_from_each_model(f=f_MW,
                                     MERGER_CRITERION=MERGER_CRITERION,
                                     MERGER_EOL=MERGER_EOL, 
                                     MERGER_MASSLOSS=MERGER_MASSLOSS,
                                     EXP_CRIT=EXP_CRIT, 
                                     KICKS=KICKS,
                                     )
GC.gc()

@printf("done\n")



In [ ]:

f = f_MW

do_file = false
do_file && (fp = open(POPSYNTH.BG_dir*"Mcore_Menv_corr_new.dat", "w"))
do_file && write(fp, @sprintf("%15s %15s %15s %15s %15s %7s\n", "Mhe", "Menv", "Mend", "Mhecore_end", "Mcocore_end", "Xc"))

keys_MODELS = sort!(collect(keys(MODELS)))
logM = 0

n = length(keys_MODELS)
mhe =zeros(Float64, n).+1e-99
menv=zeros(Float64, n).+1e-99
ccc =Array{Any}(undef, n)

for k in range(1,length(keys_MODELS))
    m = keys_MODELS[k]
    color = :black
    ccc[k] = color 

    (logM != parse(Float64, m[1:4])) && @printf("logM=%.2f\r", parse(Float64, m[1:4]))
    logM = parse(Float64, m[1:4])
    logM < 0.95 && continue
    Mi = 10^ logM

    mod = MODELS[m]
    model = mod["mod"]
    occursin("mer", mod["casus"]) && continue 
    f[!, "deltaM_A"][model] > 0 && continue 
    !(f[!, "M_he_core_1_end_BURN_He_1"][model] > 0) && continue 

    SN = (mod["1_whichstar"]=="1") ? mod["1"].SN[1] : mod["2"].SN[1] 
    (SN == :WD)                 && (color = :gray)
    (SN == :IIP)                && (color = :red)
    (SN == :Ibc || SN == :Ibn)  && (color = :blue)
    (SN == :IIb)                && (color = :green)
    (SN == :IIn)                && (color = :red)
    (SN == :X)                  && continue
    
    mhe[k]  = f[!, "M_he_core_1_ini_BURN_He_1"][model] 
    menv[k] = f[!, "M_1_ini_BURN_He_1"][model]-mhe[k]
    ccc[k] = color 

    Mend = f[!,"M_1_end_BURN_He_1"][model] 
    Mhecore_end = f[!, "M_he_core_1_end_BURN_He_1"][model]
    Mcocore_end = f[!, "M_co_core_1_end_BURN_He_1"][model]
    Xc = f[!, "Cc_1_end_BURN_He_1"][model]


    

    do_file && write(fp, @sprintf("%15.3f %15.3f %15.3f %15.3f %15.3f %7.3f\n", Mhe, Menv,
                                    Mend, Mhecore_end, Mcocore_end, Xc))





end
do_file && close(fp)


In [ ]:
using DataFrames 
d12 = CSV.read(POPSYNTH.BG_dir*"Mcore_Menv_corr_new.dat",
 DataFrame,delim=' ', ignorerepeated=true)
_logMhe=log10.(d12.Mhe)
_logMenv=log10.(d12.Menv)

FIG = Figure(size=(400,400))

grid = FIG[1,1] = GridLayout()
xticks_v=union(1:1:10, 10:10:100)
yticks_v=[.1,.25,.5,.75,1]
xticks_t = [L"%$(i)" for i in xticks_v]
yticks_t = [L"%$(@sprintf(\"%.2f\",i))" for i in yticks_v]
xticks=(xticks_v,xticks_t)
yticks=(yticks_v,yticks_t)

ax = plot_settings(grid[1,1]; 
    xlabel =    L"M_\mathrm{core} \ [M_\odot]", 
    ylabel=  L"M_\mathrm{env}   \ [M_\odot]", 
    title ="", 
    yticklabelsize=14,  
    ylabelsize=18,
    xticklabelsize=14,  
    xlabelsize=18,
    xticks=([0.3,1,3,10,30]), xmticks=union(0.1:0.1:1,1:1:10,10:10:100),
    xscale=log10,yscale=log10,
    yticks=([0.3,1,3,10,20]), ymticks=union(0.1:0.1:1,1:1:10,10:10:100))

scatter!(ax,   mhe, menv, color = ccc, markersize=8, marker = :rect)

for logm_core in 0:0.05:0.6
    @printf("%.2f ", logm_core)
    for logm_env in -.5:0.1:1.

    
        # ix = POPSYNTH.get_nearest_category_weighted_ix(logM_core,logM_env, logm_core, logm_env; yweight=1/5)
        SN, _ = POPSYNTH.get_end_from_ini_BURN_He(10 ^ logm_core, 10 ^logm_env, "X"; logMcore = _logMhe, logMenv=_logMenv, reference_grid="MW-ZAMS")
        color = "black"
        SN == :WD  && (color = "gray")
        SN == :IIP && (color = "red")
        SN == :Ibc && (color = "blue")
        SN == :IIb && (color = "green")
        
        scatter!(ax,   10^logm_core, 10^logm_env, color = color, markersize=5, marker = :rect, alpha = 0.5)

    end
end

xlims!(ax, 1.5, 4)
ylims!(ax, 0.4, 20)

# xlims!(axM, 1., 30)
# ylims!(axM, 0.4, 20)


FIG

In [ ]:
fms = CSV.read(POPSYNTH.BG_dir*"Mcore_Menv_corr_new.dat",
                 DataFrame,delim=' ', ignorerepeated=true)

Mcore = 3
Menv  = 1
ix = POPSYNTH.get_nearest_category_weighted_ix(_logMhe,_logMenv, log10(Mcore), log10(Menv); yweight=1/5)
@printf("Mcore/env queried = %5.3f/%5.3f -> got [%d] = %5.3f/%5.3f\n", Mcore, Menv,ix, d12.Mhe[ix], d12.Menv[ix])
@printf("This maps to Mend/Mhe/Mco = %5.3f/%5.3f/%5.3f\n", d12.Mend[ix], d12.Mhecore_end[ix], d12.Mcocore_end[ix])
SN, _ = POPSYNTH.get_end_from_ini_BURN_He(Mcore, Menv, "X"; logMcore = _logMhe, logMenv=_logMenv, reference_grid="MW-ZAMS", debug = true)
@printf(" -> SN = %s\n", SN)
cc = POPSYNTH.does_it_CC(Mhe = d12.Mhecore_end[ix], Mco = d12.Mcocore_end[ix])
@printf(" -> CC? %s\n", cc)

In [ ]:
using NearestNeighbors

"""
    get_nearest_category(x, y, z, xq, yq)

Given arrays `x`, `y`, and categorical values `z`, returns the category value
at the nearest `(x, y)` point to the query point `(xq, yq)`.

Arguments:
- `x`, `y` : Arrays of coordinates of the known data points
- `z`      : Array of category values (length must match `x`, `y`)
- `xq`, `yq` : Query point

Returns:
- `z_nearest`: category of the nearest data point
"""
function get_nearest_category_weighted_ix(x, y,   xq, yq; xweight=1.0, yweight=1.0)
    @assert length(x) == length(y) 

    # Scale data according to weights
    xs = x .* xweight
    ys = y .* yweight

    data = hcat(xs, ys)'
    tree = KDTree(data)

    # Scale the query point too
    xq_scaled = xq * xweight
    yq_scaled = yq * yweight

    idxs, _ = knn(tree, [xq_scaled, yq_scaled], 1)
    return idxs[1]
end


In [ ]:
d12 = CSV.read("/vol/aibn133/data1/aercolino/GRID_CACHES/hjin/Mcore_Menv_corr.dat",
 DataFrame,delim=' ', ignorerepeated=true)
Mhe=d12.Mhe
Menv=d12.Menv

get_nearest_category_ix(Mhe,Menv, 10, 5)

In [ ]:
length(0:0.025:1.5)*length(-1:0.025:1.5)


# Explore M4 mu4 in grid


In [ ]:
dir = "/vol/aibn133/data1/aercolino/GRID_CACHES/aerk/singleCC/single_star/"
for logM in 1.10:0.01:2.01
    filename = @sprintf("profCC_%4.2f.data", logM)
    prof = CSV.read(dir*filename, DataFrame, delim=' ',ignorerepeated=true, header = 6)
    s = prof.entropy
    m = prof.mass 
    r = prof.radius*Rsun

    dm = 0.3
    ix_s4 = find_nearest(s, 4)
    M4 = m[ix_s4]
    ix_dm = find_nearest(m , M4+dm)
    km_to_cm = 1e5
    mu4 = (dm)  /  ((r[ix_dm]-r[ix_s4])/(1000*km_to_cm)) 

    ks = Dict("S19.8"    =>[0.294, 0.0468],
        "S19.8_old"=>[0.274, 0.0470],
        "W20"      =>[0.273, 0.0413],
        "W20_old"  =>[0.284, 0.0393],
        "W15_old"  =>[0.225, 0.0495], 
        "W18_old"  =>[0.283, 0.0438], 
        "N20_old"  =>[0.194, 0.0580]
    )
    k=ks["N20_old"]
    x = mu4 * M4
    y = mu4
    ycrit = k[1] .* x .+ k[2]

    @printf("%4.2f : mu4 = %5.4f   M4 = %5.4f  -> M4mu4 = %5.4f : Explodes? %s \n", logM, mu4, M4, mu4*M4, y .< ycrit)

    # m_to_s = interpolator(reverse(m), reverse(s), extrapolation=:flat)
    # m_to_r = interpolator(reverse(m), reverse(r), extrapolation=:flat)
    # function newton_find_root(f, y; x0=1.0, tol=1e-8, maxiter=2000, h=1e-6)
    #     x = x0
    #     for i in 1:maxiter
    #         fx = f(x) - y
    #         if abs(fx) < tol
    #             # print("Newton converged after $i iterations.\n")
    #             return x
    #         end
    #         # Approximate derivative using central difference
    #         dfdx = (f(x + h) - f(x - h)) / (2h)
    #         if dfdx == 0.0
    #             error("Derivative is zero. Newton's method fails.")
    #         end
    #         x -= fx / dfdx
    #     end
    #     error("Newton's method did not converge after $maxiter iterations.")
    # end

    # M4_interp = nothing 
    # try 
    #     M4_interp = newton_find_root(m_to_s, 4; x0 = m[ix_s4])
    # catch 
    #     continue 
    # end
    # R_M4 = m_to_r(M4_interp)
    # R_M4pdm = m_to_r(M4_interp+dm)
    # mu4_interp=  dm  /  ((R_M4pdm-R_M4)/(1000*km_to_cm))

    # @printf("%4.2f : mu4 = %5.3f (%5.1f%%)   M4 = %5.3f  (%5.1f%%) -> M4mu4 = %5.3f  (%5.1f%%)\n", logM, mu4_interp, (mu4-mu4_interp)/mu4_interp*100, M4_interp, (M4-M4_interp)/M4_interp*100, mu4_interp*M4_interp, (mu4*M4-mu4_interp*M4_interp)/(mu4_interp*M4_interp)*100)
end